In [49]:
import math
import random

In [337]:
class Mathlib:
    @staticmethod
    def clip(val, minVal=0, maxVal=1):
        return(min(maxVal, max(minVal, val)))
    
    @staticmethod
    def clipAboveZeroBelowOne(val):
        """ clip between 1e-7 and 1-1e-7 """
        return(Mathlib.clip(val, 1e-7, 1-1e-7))

    @staticmethod
    def mean(vals):
        return(sum(vals)/len(vals))
    
    @staticmethod
    def argmax(vals):
        maxVal = vals[0]
        maxIndex = 0
        for i, v in enumerate(vals):
            if v > maxVal:
                maxVal = v
                maxIndex = i
        return(maxIndex)
    
    @staticmethod
    def argmin(vals):
        minVal = vals[0]
        minIndex = 0
        for i, v in enumerate(vals):
            if v < minVal:
                minVal = v
                minIndex = i
        return(minIndex)

    @staticmethod
    def dot2Vectors(vector1, vector2):
        out = 0
        for v1, v2 in zip(vector1, vector2):
            out += v1*v2
        return(out)
    
    @staticmethod
    def dotVectorMatrix(vector, matrix):
        out = []
        for row in matrix:
            out.append(Mathlib.dot2Vectors(vector, row))
        return(out)

    @staticmethod
    def addTwoVectors(v1, v2):
        return([i1+i2 for i1,i2 in zip(v1,v2)])

    @staticmethod
    def transpose(m):
        return(list(zip(*m)))

    @staticmethod
    def elementWiseMult(v1, v2):
        result = []
        for i1, i2 in zip(v1, v2):
            result.append(i1*i2)
        return(result)

    @staticmethod
    def normalize(values):
        out = []
        total = sum(values)
        for v in values:
            out.append(v/total)
        return(out)

    @staticmethod
    def randomNumber(minimum=-10.0, maximum=10.0, decimals=2):
        return(round(random.uniform(minimum, maximum), decimals))

#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



In [ ]:
class Activation:

    class Pass:
        """ returns x """
        @staticmethod
        def forward(val):
            return(val)

        @staticmethod
        def backward(*args):
            """ derivative of x with respect to x """
            return(1)

    class ReLU:
        """ Returns 0 if values and x if its not """
        @staticmethod
        def forward(val):
            return(max(0, val))    #< by clipping 0, you de-linearize your data
        
        @staticmethod
        def backward(val):
            """ partial derivative of max(x,0) with respect to x """
            return(1 if val > 0 else 0)

    class Softmax:
        """ converts any output to be squashed from 0 to 1 and also had a nice derivative when paired with  Cross-Entropy """
        @staticmethod
        def forward(vals):
            vals = [math.e**val for val in vals]
            vals = Mathlib.normalize(vals)
            return(vals)
        
        @staticmethod
        def backward(vals):
            return(Activation.Softmax.backward_jacobian(vals))

        @staticmethod
        def backward_jacobian(vals):
            """ derivative of e^x/(sum(e^x) for x in vals) with respect to x
            For proof, see image above or in 'proofs_math/ActivationFuncs'"""
            jacobian = []
            for i, val in enumerate(vals):
                row = []
                for j in range(len(vals)):
                    if j == i:
                        row.append(val*(1-val))
                    else:
                        row.append(-val*vals[j])
                jacobian.append(row)
            return(jacobian)
        
        @staticmethod
        def backward_crossEntropy(vals):
            pass

    class ProtectedSoftmax:
        """ Softmax but without any overflow from e^x being too large """
        @staticmethod
        def forward(vals):
            """ softmax, but between 0 and 1 """
            maxVal = max(vals)
            return(Activation.Softmax.forward([val-maxVal for val in vals]))
        
        @staticmethod
        def backward(vals):
            """ derivative of e^x with respect to x as protected softmax is just regular softmax with extra steps to ensure values are normalized """
            return(Activation.Softmax.backward(vals))


#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

In [344]:
class Loss:
    """ Calculates the error of the output (mainly Softmax) using -log(x) """
    @staticmethod
    def forward_single(val):
        """ return the loss of the function, caped at a minimum of 16.12 and a 0.00000001 for division purposes"""
        return(-math.log(Mathlib.clipAboveZeroBelowOne(val)))  #< math.log is actually ln

    @staticmethod
    def forward(vals, targets):
        """calculate the cross entropy loss of a softmax output

        Args:
            vals (list): results
            targets (list): target results
        """
        out = 0
        for v, t in zip(vals, targets):
            out += t*Loss.forward_single(v)   #< loss times the weight of the loss, more important = bigger loss
        return(out)
    
    @staticmethod
    def forwardHot(vals, target):
        """ Forward method optimized for a hot target vector """
        for t, val in zip(target, vals):
            if t == 1:
                return(Loss.forward_single(val))
        return(Loss.forward_single(0))

    @staticmethod
    def forward_batch(vals, targets):
        return(Loss.forward(vals, targets)/len(vals))                 #< mean of the losses

    @staticmethod
    def backward(vals, targets):
        return([-t / Mathlib.clipAboveZeroBelowOne(v) for v, t in zip(vals, targets)])

    @staticmethod
    def backwardHot(vals, targets):
        """ Backward method optimized for a hot target vector """
        out = []
        for v, t in zip(vals, targets):
            if t == 1:
                out.append(-1 / Mathlib.clipAboveZeroBelowOne(v))
            else:
                out.append(0)
        return(out)


In [180]:
class Accuracy_hard:
    """ calculates how accurate the the result is, for the hard version, it gets an array of index (must be all 0 and a single value has to be 1), and it returns 1 if that most likely output is at the correct index and 0 if its not """
    def __init__(self):
        self._length = 0
        self.correct = 0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        if type(target) == list:
            target = Mathlib.argmax(target)
        if Mathlib.argmax(result) == target:
            return(1)
        return(0)

    def epoch(self, result, target):
        self.correct += Accuracy_hard.calc(result, target)
        self._length += 1
        self.accuracy = self.correct/self._length
        return(self.accuracy)

class Accuracy_soft:
    """ calculates how accurate the the result is, for the soft version, it get an array of target results, ex: [0.2, 0.8, 0.0, 0.0] and returns a number to quantify how close it is"""
    def __init__(self):
        self._length = 0
        self.correct = 0.0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        dot = Mathlib.dot2Vectors(result, target)
        norm_r = math.sqrt(sum(r*r for r in result))
        norm_t = math.sqrt(sum(t*t for t in target))
        soft = dot / (norm_r * norm_t)      #< normalize between 0 and 1
        return(soft)

    def epoch(self, result, target):
        soft = Accuracy_soft.calc(result, target)
        self._length += 1
        self.correct += soft
        self.accuracy = self.correct/self._length
        return(self.accuracy)
        

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



In [ ]:
class Neuron:
    """ A single neuron, stores its inputs, weights, bias, and activation function """
    def __init__(self, inputs, weights, bias, activationFunc=Activation.Pass):
        self.inputs = inputs      #< all outputs from previous layer
        self.weights = weights    #< one weight for every input
        self.bias = bias          #< one bias per neuron

        self.out = None
        self.d_inputs = None
        self.d_weights = None
        self.d_bias = None


        self.activationFunc = activationFunc

        if len(self.weights) != len(self.inputs):
            raise Exception("inputs and weight must have the same length!")

    def forward(self):
        """
        Compute the neuron's output.
        Multiply each input by its corresponding weight, sum the
        results, add the bias and then apply the activation function.
        
        multiply the inputs [i1, i2.. in] with weights [w1, w1... wn] to get i1*w1+i2*w2... +in*wn, then add the bias
        """
        self.out = Mathlib.dot2Vectors(self.inputs, self.weights)+self.bias
        return(self.activationFunc.forward(self.out))

    def backward(self, d_val):
        """ Compute gradient 
        since forward is computed as Activation(sum(x1*w1, x2*w2..., bias)),
        the backward for the weights would be computed as [Activation`(...)*sum`(...)*xi] for every element,
        and the backward for the inputs would be computed as [Activation`(...)*sum`(...)*wi] for every element,
        note that sum`(...) is equal to 1 no matter the reference or input.
        """
        activation_dx = self.activationFunc.backward(self.out)*d_val #< chain rule
        self.d_inputs = [activation_dx*w for w in self.weights]
        self.d_weights = [activation_dx*i for i in self.inputs]
        self.d_bias = activation_dx
        return(self.d_inputs, self.d_weights, self.d_bias)


In [335]:
class Layer:
    """A collection of neurons forming a single layer in the network"""
    def __init__(self, inputs, weights, biases,
                 activationFunc=Activation.Pass,   #< Neuron-level activation function
                 *args, **kwargs):
        
        self.inputsLen = len(inputs)
        self.neurons = self._createNeurons(inputs, weights, biases, activationFunc=activationFunc, *args, **kwargs)  # create neuron objects
        self.activationFunc = activationFunc
        self._resetLayer_dx()

    def _resetLayer_dx(self):
        self.d_inputs = [0]*self.inputsLen
    
    def _createNeurons(self, inputs, weights, biases, activationFunc, *args, **kwargs):
        neurons = []
        for w, b in zip(weights, biases):
            neurons.append(Neuron(inputs, w, b, activationFunc, *args, **kwargs))
        return(neurons)

    def forward(self):
        """Calculate outputs for all neurons in the layer """
        return([neuron.forward() for neuron in self.neurons])
    
    def backward(self, d_values):
        """Calculate gradients for all neurons in the layer."""
        self._resetLayer_dx()
        for neuron, d_val in zip(self.neurons, d_values):
            res = neuron.backward(d_val)
            self.d_inputs = Mathlib.addTwoVectors(res[0], self.d_inputs)   #< this is done as an optimization, every neuron CAN adjust its input, but since they are all going to adjust the same inputs, we can sum up their adjustments and adjust it in one go
        return(self.d_inputs)


In [167]:
class Batch:
    """ Process multiple samples at once in a batch."""
    def __init__(self, inputsBatch, *args, **kwargs):
        self.batch = self._createBatch(inputsBatch, *args, **kwargs)  # create a Layer for each sample

    def _createBatch(self, inputsBatch, *args, **kwargs):
        """Helper method to create a Layer object for each sample in the batch."""
        batch = []
        for inputs in inputsBatch:
            batch.append(Layer(inputs, *args, **kwargs))  # instantiate Layer and add to batch
        return(batch)
    
    def forward(self):
        """Run all layers in the batch and return their outputs."""
        return([layer.forward() for layer in self.batch])
    
    def calcLoss(self, correctIndexes):
        """Compute mean loss across the batch given a list of target indexes."""
        return(Mathlib.mean([self.batch[i].calcLoss(correctIndexes[i]) for i in range(len(self.batch))]))  # average of per-sample losses


In [168]:
class Train:
    """ TBD """
    def __init__(self, weights, biases, learningRate=0.01):
        self.weights = weights
        self.biases = biases
        self.learningRate = learningRate

    def step(self, inputs, correctIndex):
        """Perform a single gradient descent update for one example.

        Args:
            inputs: list of input features for the sample
            correctIndex: index of the true class in the output vector
        """
        layer = Layer(inputs, self.weights, self.biases)
        out = layer.forward()
        for i in range(len(self.weights)) :
            # target probability is 1 for correct class, 0 otherwise (eg: 0 for cat, 0 for dog, 1 for horse)
            target = 0
            if i == correctIndex:
                target = 1
  
            # protectedSoftmax produces an output of e^x/(sum(e^x))
            # the error function is -log(result)
            # the error is -sum(targetX*log(e^x/(sum(e^x)))
            # for target with 0s and 1s [0, 0, 1, 0], the error simplifies to -log(e^x/(sum(e^x))
            # to know if the error has improved, we need to take the derivative of the error
            # the derivative of -log(e^x/(sum(e^x)) simplifies to result-target
            error = out[i] - target
            for j in range(len(self.weights[i])):
                # learning rate determines the size of the change
                # the error determines how far away we are, therefore the size of the update
                # inputs determines the importance of that input
                # an smaller input that has less effect, will be changed less
                self.weights[i][j] -= self.learningRate * error * inputs[j]
            # learning rate determines the size of the change
            # the error determines how far away we are, therefore the size of the update
            # since we update both the weights and biases, we might overcompensate, which is ok because there are multiple iterations
            self.biases[i] -= self.learningRate * error

    def train(self, inputsBatch, correctIndexes, epochs=10):
        """Train for a number of epochs """
        for _ in range(epochs):
            for inputs, correct in zip(inputsBatch, correctIndexes):
                self.step(inputs, correct)

## Function that will be used to create datasets

In [312]:
def createInputs(inputCount=10):
    return([Mathlib.randomNumber() for _ in range(inputCount)])

def createLayerData(neuronCount=5, inputCount=10):
    weights = [[Mathlib.randomNumber(minimum=-1.0, maximum=1.0) for _ in range(inputCount)] for _ in range(neuronCount)]
    biases = [Mathlib.randomNumber() for _ in range(neuronCount)]
    return(weights, biases)

## Now lets run a forward pass

In [349]:
random.seed(1)

inputsCount = 5
targets = [0, 0, 1, 0, 0]
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
weights1, biases1 = createLayerData(neuronCount=layer1NeuronCount, inputCount=inputsCount)
layer1 = Layer(inputs, weights1, biases1, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
layer1Output = layer1.forward()

layer2NeuronCount = 5
weights2, biases2 = createLayerData(neuronCount=layer2NeuronCount, inputCount=len(layer1Output))  #< layer1Output is the input for the next layer
layer2 = Layer(layer1Output, weights2, biases2, activationFunc=Activation.ReLU)   #< weights/biases determine neuron count
layer2Output = layer2.forward()

softmaxOutput = Activation.Softmax.forward(layer2Output)
error = Loss.forwardHot(softmaxOutput, targets)

## seeing model accuracy
accuracy = Accuracy_hard.calc(layer2Output, targets)    #< we want to maximize the 2nd output
print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
print(f"WEIGHTS: {weights1}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
print(f"BIASES: {biases1}\n    this layer should have {layer1NeuronCount} biases")
print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
print()
print(f"WEIGHTS: {weights2}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
print(f"BIASES: {biases2}\n    this layer should have {layer2NeuronCount} biases")
print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
print("SOFTMAX: ", softmaxOutput)
print("TARGET: ", targets)
print()
print()
print("ERROR: ", error)
print("ACCURACY: ", accuracy)


INPUTS: [-7.31, 6.95, 5.28, -4.9, -0.09]
    This layer should have 5 inputs
WEIGHTS: [[-0.1, 0.3, 0.58, -0.81, -0.94], [0.67, -0.13, 0.52, -1.0, -0.11], [0.44, -0.54, 0.89, 0.8, -0.94], [-0.95, 0.08, 0.88, -0.24, -0.57]]
    this layer should have 4 weights with 5 values in them
BIASES: [-1.56, -9.42, -5.57, -1.24]
    this layer should have 4 biases
RESULT: [8.372, 0, 0, 12.134199999999998]
     this layer should have 4 outputs

WEIGHTS: [[-0.01, -0.53, -0.54, -0.56], [-0.08, -0.42, -0.96, 0.68], [0.11, 0.28, -0.63, 0.99], [0.72, -0.76, -0.33, 0.44], [0.42, 0.87, -0.16, 0.66]]
    this layer should have 5 weights with 4 values in them
BIASES: [3.41, -3.93, 1.75, 7.65, 6.92]
    this layer should have 5 biases
RESULT: [0, 3.6514959999999994, 14.683777999999998, 19.016888, 18.444812]
     this layer should have 5 outputs
SOFTMAX:  [3.492261479035917e-09, 1.3456475252761318e-07, 0.008321287218754293, 0.6339226581263162, 0.35775591659791545]
TARGET:  [0, 0, 1, 0, 0]


ERROR:  4.788938322

### Now lets run a manual backward pass, calculating and storing every derivative, and then apply the chain rule ourself
* Later we can show how to improve this by passing the gradient to the next layer and allowing it to compute the chain rule itself

In [350]:
d_error  = Loss.backwardHot(softmaxOutput, targets)
print("d_ERROR: ", d_error)
d_softmax = Activation.Softmax.backward(layer2Output)
print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(layer2Output)}")
d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax)
print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
d_layer2 = layer2.backward(d_crossEntropy)
print("d_LAYER 2: ", d_layer2)
d_layer1 = layer1.backward(d_layer2)
print("d_LAYER 1: ", d_layer1)

d_ERROR:  [0, 0, -120.17371516106631, 0, 0]
d_SOFTMAX: [[0, 0.0, 0.0, 0.0, 0.0], [-0.0, -9.681927038015996, -53.617756631887985, -69.44009046444799, -67.35115723875198], [-0.0, -53.617756631887985, -200.92955835328397, -279.239761642864, -270.83952465973596], [-0.0, -69.44009046444799, -279.239761642864, -342.62514120454404, -350.76292398505603], [-0.0, -67.35115723875198, -270.83952465973596, -350.76292398505603, -321.76627771534396]]
    this layer should be 5x5
d_CROSS_ENTROPY: [0.0, 6443.445013055881, 24146.4515129864, 33557.27957731359, 32547.791890817705]
    this layer should have 5 outputs
d_LAYER 2:  [39971.94795519325, 6867.805984405801, -37679.5206287594, 64533.2752686922]
d_LAYER 1:  [-65303.806300776916, 17154.246408053354, 79973.01205046123, -47865.26390819266, -74357.5979810362]
